[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C14_DL_Theory_Data_Course/01_generalization_double_descent/01_generalization_double_descent.ipynb)

# 01 · 泛化与双下降（用 numpy 从零复现）

目标：把 **bias-variance 分解**、**经典 U 型**、**插值阈值尖峰**、**double descent**、**最小范数解/隐式正则** 全部用纯 numpy 复现，并用 `assert` 钉死每条结论。

路线：bias-variance 分解(多项式回归) → 经典 U 型 → 随机特征 + 容量扫描 → 复现 double descent 曲线 → 最小范数解 vs 任意插值解 → ridge 抹平尖峰 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(加州房价随机特征双下降)。

> 心智模型：**容量不是越大越过拟合；越过插值阈值后，最小范数解越平滑、测试越好**。我们要把这条非单调曲线*亲手画出来*。

## 1 · 造一个我们知道全部真相的回归问题

真函数取一个固定的非线性曲线，加**标签噪声**（双下降的尖峰本质来自过度拟合噪声，无噪声尖峰很弱）。训练集小、测试集大且独立。一切结论都能对照 ground truth 验证。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def true_f(x):
    '''固定真函数(我们知道真相)。'''
    return np.sin(2.0 * x) + 0.5 * x

def make_data(n, noise=0.3, seed=0, lo=-3.0, hi=3.0):
    rng_ = np.random.default_rng(seed)
    x = rng_.uniform(lo, hi, size=n)
    y = true_f(x) + noise * rng_.standard_normal(n)   # 含标签噪声
    return x, y

x_tr, y_tr = make_data(25, noise=0.3, seed=1)
x_te, y_te = make_data(500, noise=0.0, seed=999)      # 测试集用无噪真值, 测的是逼近真函数的能力
print(f'训练集 {x_tr.shape[0]} 点(含噪), 测试集 {x_te.shape[0]} 点(真函数)')
print(f'真函数在 x=1 处 = {true_f(1.0):.4f}')
assert x_tr.shape == (25,) and x_te.shape == (500,)
assert abs(true_f(0.0) - 0.0) < 1e-9
print('✅ 数据就绪: 小训练集(含噪) + 大独立测试集(真值)')

## 2 · 偏差-方差分解：在多套训练集上重复拟合

**方差只能通过「在多套独立训练集上重复拟合」来测量**。对每个多项式次数 `k`：
- 跑 `R` 次（每次新采一套训练集），在固定测试点上得到 `R` 条预测；
- **bias² = (R 次预测的均值 − 真值)²** 的平均；**variance = R 次预测的离散度** 的平均。

用 `np.polyfit`/`np.polyval`（最小二乘多项式拟合）当模型。

In [ ]:
def fit_poly_predict(x_tr, y_tr, x_eval, k):
    '''次数 k 的多项式最小二乘拟合, 在 x_eval 上预测。'''
    coef = np.polyfit(x_tr, y_tr, deg=k)
    return np.polyval(coef, x_eval)

def bias_variance(k, n_train=25, R=200, noise=0.3):
    '''在 R 套独立训练集上拟合次数 k, 估计 bias^2 与 variance(在一组测试点上平均)。'''
    x_eval = np.linspace(-2.5, 2.5, 50)
    truth = true_f(x_eval)
    preds = np.zeros((R, x_eval.shape[0]))
    for r in range(R):
        xt, yt = make_data(n_train, noise=noise, seed=1000 + r)
        preds[r] = fit_poly_predict(xt, yt, x_eval, k)
    mean_pred = preds.mean(axis=0)
    bias2 = np.mean((mean_pred - truth) ** 2)          # 平均后还剩的系统误差
    variance = np.mean(preds.var(axis=0))              # 各次拟合的抖动
    return bias2, variance

for k in [1, 3, 7]:
    b2, v = bias_variance(k)
    print(f'次数 k={k}: bias^2={b2:.4f}  variance={v:.4f}')
b2_lo, v_lo = bias_variance(1)
b2_hi, v_hi = bias_variance(9)
assert b2_lo > b2_hi, '低次偏差应更大(欠拟合)'
assert v_hi > v_lo, '高次方差应更大(过拟合抖动)'
print('✅ 偏差随次数下降、方差随次数上升 —— 经典权衡的两半被分别测出')

## 3 · 经典 U 型：测试误差先降后升

把 bias² + variance 合起来扫一遍次数（**欠参数化区**，次数 < 训练点数），测试误差应呈 **U 型**。
这是经典理论最得意的预言——也是双下降要超越的起点。

In [ ]:
def test_mse_poly(k, n_train=25, R=200, noise=0.3):
    '''次数 k 在独立测试集上的平均测试 MSE(对 R 套训练集平均)。'''
    x_eval = np.linspace(-2.5, 2.5, 200)
    truth = true_f(x_eval)
    errs = []
    for r in range(R):
        xt, yt = make_data(n_train, noise=noise, seed=2000 + r)
        pred = fit_poly_predict(xt, yt, x_eval, k)
        errs.append(np.mean((pred - truth) ** 2))
    return np.mean(errs)

degrees = list(range(1, 13))
mses = [test_mse_poly(k) for k in degrees]
print(f"{'次数':>4}{'测试MSE':>12}")
for k, m in zip(degrees, mses):
    bar = '#' * int(min(m, 5) * 8)
    print(f'{k:>4}{m:>12.4f}  {bar}')
best_k = degrees[int(np.argmin(mses))]
print(f'最优次数 ≈ {best_k}')
assert mses[0] > mses[best_k - 1], 'U 型左臂: 低次误差更高'
assert mses[-1] > mses[best_k - 1], 'U 型右臂: 高次误差回升'
assert 1 < best_k < 12, 'U 型最优在中间'
print('✅ 经典 U 型成立: 测试误差先降后升, 最优在中间次数')

## 4 · 随机特征回归：把容量做成连续旋钮

要画出**完整**双下降(含越过阈值的第二段), 用**随机特征**: 一个 `d` 维输入经**固定随机** `W` 升维成 `φ=relu(Wx+b)`, **特征数 p = 容量旋钮**, 在 `φ` 上做最小二乘 / 最小范数回归。

(为什么换成 d=8 维输入? 一维输入的 cos/relu 特征高度共线、拟合不满秩, 画不出尖峰; 真实的随机特征双下降需要输入本身有足够维度。)

拟合规则: `np.linalg.lstsq` 在 `p<n` 时给普通最小二乘解, 在 `p≥n`(过参数化) 时自动给**最小范数**插值解。

In [ ]:
d = 8
# 固定的非线性 teacher(我们知道真相): y = tanh(Xw*) + 0.5 Xw*
w_true = np.random.default_rng(123).standard_normal(d) / np.sqrt(d)

def make_xy(n, noise, seed):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, d))
    sig = X @ w_true
    y = np.tanh(sig) + 0.5 * sig + noise * r.standard_normal(n)
    return X, y

Xtr, Ytr = make_xy(40, noise=0.5, seed=1)        # 训练: 含标签噪声
Xte, Yte = make_xy(400, noise=0.0, seed=999)     # 测试: 真函数(测逼近能力)
n = Xtr.shape[0]
print(f'输入维度 d={d}, 训练 n={n}(含噪), 测试 {Xte.shape[0]}(真值)')

def relu(z):
    return np.maximum(z, 0.0)

def random_features(X, W, b):
    '''X:(n,d), W:(d,p), b:(p,) -> φ:(n,p), 固定随机映射 + ReLU 非线性。'''
    return relu(X @ W + b)

def fit_min_norm(Phi, y):
    '''最小二乘 / 最小范数解: 欠定时 lstsq 返回范数最小的插值解。'''
    w, *_ = np.linalg.lstsq(Phi, y, rcond=None)
    return w

def test_err_rf(p, seed=0):
    '''返回 (训练MSE, 测试MSE)。特征数 p = 容量。'''
    r = np.random.default_rng(seed)
    W = r.standard_normal((d, p)) / np.sqrt(d)
    b = r.standard_normal(p) * 0.1
    Phi_tr = random_features(Xtr, W, b)
    Phi_te = random_features(Xte, W, b)
    w = fit_min_norm(Phi_tr, Ytr)
    return np.mean((Phi_tr @ w - Ytr) ** 2), np.mean((Phi_te @ w - Yte) ** 2)

tr5, te5 = test_err_rf(5)
tr200, te200 = test_err_rf(200)
print(f'p=5   (欠参数化): train={tr5:.3f}  test={te5:.3f}')
print(f'p=200 (过参数化): train={tr200:.2e}  test={te200:.3f}')
assert tr200 < 1e-6, '过参数化(p≫n)应能完美插值训练集(训练误差≈0)'
assert tr5 > tr200, '欠参数化拟合不了训练集'
print('✅ 随机特征就绪: p≥n 时训练误差→0(插值), 特征数 p 就是容量旋钮')

## 5 · 复现 double descent 曲线

沿 `p` 从小到大扫过插值阈值 `p≈n=40`, 对随机 `W` 多次取**中位数**降噪(尖峰附近误差重尾, 中位数比均值稳健), 记录测试误差。
**预言: 测试误差在 `p≈n` 处冲到尖峰, 之后随 p 增大二次下降到比欠参数化还低。** 这就是 Belkin 2019 的核心图。

In [ ]:
def double_descent_curve(p_list, n_seeds=15):
    test_curve = []
    for p in p_list:
        tes = [test_err_rf(p, seed=s)[1] for s in range(n_seeds)]
        test_curve.append(np.median(tes))     # 中位数: 对尖峰附近的重尾更稳健
    return np.array(test_curve)

p_list = [5, 15, 25, 35, 40, 45, 55, 70, 100, 150, 250, 400]
curve = double_descent_curve(p_list)
print(f"{'p(容量)':>8}{'测试MSE(中位)':>14}{'p/n':>7}")
for p, m in zip(p_list, curve):
    flag = ' <- 插值阈值附近' if abs(p - n) <= 5 else ''
    bar = '#' * int(min(m, 3) * 10)
    print(f'{p:>8}{m:>14.3f}{p/n:>7.2f}  {bar}{flag}')

peak_idx = int(np.argmax(curve))
peak_p = p_list[peak_idx]
print(f'\n尖峰出现在 p={peak_p} (n={n}, 比值 {peak_p/n:.2f})')
assert abs(peak_p - n) <= 10, '尖峰应在插值阈值 p≈n 附近'
assert curve[-1] < curve[peak_idx], '过参数化区(大p)测试误差应远低于尖峰 -> 二次下降'
assert curve[-1] < curve[0], '极大容量应优于极小容量(第二段降到更低)'
print('✅ DOUBLE DESCENT 复现成功: 尖峰在 p≈n, 之后二次下降到比欠参数化更低')

## 6 · 为什么过参数化更好: 最小范数解 vs 任意插值解

过参数化区有**无穷多**插值解。验证: (a) 最小范数解范数确实最小; (b) 在训练特征**零空间**方向上加扰动得到另一个插值解, 它范数更大、测试更差。
**这就是隐式正则的机制**: 优化器(从0出发的GD)隐式选了那个平滑(小范数)的解。

In [ ]:
p = 200                                # 过参数化(p > n)
r = np.random.default_rng(0)
W = r.standard_normal((d, p)) / np.sqrt(d)
b = r.standard_normal(p) * 0.1
Phi_tr = random_features(Xtr, W, b)
Phi_te = random_features(Xte, W, b)

w_min = fit_min_norm(Phi_tr, Ytr)      # 最小范数插值解
# 另一个插值解 = 最小范数解 + 训练特征零空间方向(不改训练预测)
U, S, Vt = np.linalg.svd(Phi_tr, full_matrices=True)
rank = int(np.sum(S > 1e-8))
null_basis = Vt[rank:]                 # 行空间维度=rank, 其余是零空间
w_other = w_min + 5.0 * null_basis[0]

tr_min = np.mean((Phi_tr @ w_min - Ytr) ** 2)
tr_oth = np.mean((Phi_tr @ w_other - Ytr) ** 2)
te_min = np.mean((Phi_te @ w_min - Yte) ** 2)
te_oth = np.mean((Phi_te @ w_other - Yte) ** 2)
print('两解都插值训练集吗?')
print(f'  最小范数解 训练MSE = {tr_min:.2e}')
print(f'  另一插值解 训练MSE = {tr_oth:.2e}')
print(f'范数:   最小范数 {np.linalg.norm(w_min):.3f}  vs 另一解 {np.linalg.norm(w_other):.3f}')
print(f'测试MSE: 最小范数 {te_min:.3f}  vs 另一解 {te_oth:.3f}')
assert tr_min < 1e-6 and tr_oth < 1e-6, '两者都应插值训练集'
assert np.linalg.norm(w_min) < np.linalg.norm(w_other), '最小范数解范数最小'
assert te_min < te_oth, '最小范数解测试更好'
print('✅ 同样插值训练集, 最小范数解范数更小、测试更好 —— 隐式正则的因果机制')

---
## ✏️ 练习 1：偏差-方差分解

从零实现偏差-方差估计 `bias_var_at(model_fn, k, x0, R)`：在 `R` 套独立训练集上拟合, 在**单个**测试点 `x0` 估计 `(bias², variance)`。

（提示: 收集 R 条在 x0 的预测; bias² = (均值 − true_f(x0))²; variance = 这 R 条预测的方差。）

In [ ]:
def bias_var_at(k, x0, R=300, n_train=25, noise=0.3):
    # TODO: 跑 R 次(每次 make_data(seed=3000+r)), 拟合次数 k, 记录在 x0 的预测
    #       返回 (bias2, variance):
    #         bias2 = (preds.mean() - true_f(x0))**2
    #         variance = preds.var()
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
b2_1, v_1 = bias_var_at(1, x0=1.0)      # 直线: 高偏差低方差
b2_9, v_9 = bias_var_at(9, x0=1.0)      # 高次: 低偏差高方差
print(f'k=1 在 x0=1: bias^2={b2_1:.4f} var={v_1:.4f}')
print(f'k=9 在 x0=1: bias^2={b2_9:.4f} var={v_9:.4f}')
assert b2_1 > b2_9, '低次偏差应更大'
assert v_9 > v_1, '高次方差应更大'
assert b2_1 >= 0 and v_1 >= 0, '两项非负'
print('✅ 练习 1 通过: 偏差-方差在单点上的此消彼长被复现')

## ✏️ 练习 2：定位插值阈值

实现 `interpolation_threshold(p_list, n_train)`：扫容量 `p`, 返回**训练误差首次降到 ≈0** 的最小 `p`(插值阈值)。
用 `test_err_rf` 的训练误差那一项(返回元组第 0 个)。

In [ ]:
def interpolation_threshold(p_list, tol=1e-6, seed=0):
    # TODO: 对每个 p 算 test_err_rf(p, seed)[0] (训练MSE),
    #       返回第一个使训练MSE < tol 的 p; 都不满足则返回 p_list[-1]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
thr = interpolation_threshold([10, 20, 35, 40, 45, 55, 80])
print(f'插值阈值(训练误差首次≈0) p = {thr}, 训练集 n = {n}')
assert thr >= n, '要插值 n 个点, 容量至少 ≈ n'
assert thr <= n + 10, '阈值不应过分大于 n'
# 阈值及以上能插值, 阈值以下不能
assert test_err_rf(thr, 0)[0] < 1e-6, '阈值处应已插值'
assert test_err_rf(n - 15, 0)[0] > 1e-6, '阈值以下不能插值'
print('✅ 练习 2 通过: 训练误差在 p≈n 处首次归零 —— 这就是插值阈值')

## ✏️ 练习 3：容量扫描看二次下降

实现 `second_descent_check(p_small, p_peak, p_large)`：返回三个容量点的测试误差, 并验证 **峰值 > 两侧**(双下降的定性特征)。
复用 `double_descent_curve`(对单点也能调)。

In [ ]:
def second_descent_check(p_small, p_peak, p_large, n_seeds=20):
    # TODO: 用 double_descent_curve([p_small, p_peak, p_large], n_seeds)
    #       返回 (e_small, e_peak, e_large)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
e_small, e_peak, e_large = second_descent_check(5, 40, 400)
print(f'测试MSE: p=5 -> {e_small:.3f}, p=40(≈n) -> {e_peak:.3f}, p=400 -> {e_large:.3f}')
assert e_peak > e_large, '峰值应高于大容量(二次下降)'
assert e_peak > e_small, '峰值应高于小容量'
assert e_large < e_small, '极大容量应优于极小容量'
print('✅ 练习 3 通过: 容量扫描的「峰值高于两侧」就是双下降的指纹')

## ✏️ 练习 4：最小范数解

对一个**欠定**系统 `Aw=y`(行<列), 实现两个函数并验证最小范数性质:
- `min_norm_solution(A, y)`: 用伪逆 `A⁺y = Aᵀ(AAᵀ)⁻¹y` 给出最小范数解;
- `is_min_norm(A, y, w)`: 验证 `w` 插值 `Aw≈y` 且其范数 ≤ 任意「w + 零空间方向」。

In [ ]:
def min_norm_solution(A, y):
    # TODO: 行满秩欠定时, A^+ y = A.T @ inv(A @ A.T) @ y
    raise NotImplementedError

def interpolates(A, y, w, tol=1e-8):
    # TODO: 返回 Aw 是否 ≈ y
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rng2 = np.random.default_rng(7)
A = rng2.standard_normal((4, 12))      # 4 方程 12 未知 -> 欠定
yv = rng2.standard_normal(4)
w_mn = min_norm_solution(A, yv)
assert interpolates(A, yv, w_mn), '最小范数解应插值'
# 与 numpy 的 lstsq 一致
w_ref, *_ = np.linalg.lstsq(A, yv, rcond=None)
assert np.allclose(w_mn, w_ref, atol=1e-6), '应与 lstsq 的最小范数解一致'
# 加零空间方向得到的另一插值解范数更大
_, _, Vt = np.linalg.svd(A)
w_pert = w_mn + 2.0 * Vt[-1]
assert interpolates(A, yv, w_pert), '加零空间方向仍插值'
assert np.linalg.norm(w_mn) < np.linalg.norm(w_pert), '最小范数解范数最小'
print('✅ 练习 4 通过: 伪逆给出最小范数插值解, 与 lstsq 一致且范数最小')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bias_var_at(k, x0, R=300, n_train=25, noise=0.3):
    preds = np.empty(R)
    for r in range(R):
        xt, yt = make_data(n_train, noise=noise, seed=3000 + r)
        preds[r] = np.polyval(np.polyfit(xt, yt, k), x0)
    bias2 = (preds.mean() - true_f(x0)) ** 2
    variance = preds.var()
    return bias2, variance

In [ ]:
# 练习 2 参考答案
def interpolation_threshold(p_list, tol=1e-6, seed=0):
    for p in p_list:
        if test_err_rf(p, seed)[0] < tol:
            return p
    return p_list[-1]

In [ ]:
# 练习 3 参考答案
def second_descent_check(p_small, p_peak, p_large, n_seeds=20):
    e = double_descent_curve([p_small, p_peak, p_large], n_seeds)
    return float(e[0]), float(e[1]), float(e[2])

In [ ]:
# 练习 4 参考答案
def min_norm_solution(A, y):
    return A.T @ np.linalg.inv(A @ A.T) @ y

def interpolates(A, y, w, tol=1e-8):
    return np.allclose(A @ w, y, atol=tol)

---
## 🧪 真实数据胶囊：在真实数据上复现双下降

用真实数据(尝试联网拉**加州房价**, 失败则回退到内置的真实统计量构造的等价数据)做随机特征回归, 看双下降是否在真实数据上也出现。**真实数据 + 标签噪声 + 容量扫描 = 同样的尖峰。**

In [ ]:
def load_california_subset(n=40, d=6, seed=0):
    '''尝试 sklearn 加州房价; 失败则用内置真实统计量(各特征均值/尺度近似真实)生成等价数据。'''
    try:
        from sklearn.datasets import fetch_california_housing
        data = fetch_california_housing()
        X, y = data.data, data.target
        rng_ = np.random.default_rng(seed)
        idx = rng_.choice(X.shape[0], size=n, replace=False)
        X = X[idx, :d]; y = y[idx]
        src = 'sklearn 加州房价(真实)'
    except Exception:
        # 回退: 用加州房价各特征的真实量级(收入~3.9, 房龄~28, 房间数~5.4 ...)构造
        rng_ = np.random.default_rng(seed)
        means = np.array([3.87, 28.6, 5.43, 1.10, 1425., 3.07])[:d]
        scales = np.array([1.9, 12.6, 2.5, 0.47, 1132., 10.4])[:d]
        X = rng_.standard_normal((n, d)) * scales + means
        true_w = rng_.standard_normal(d)
        y = (X - means) / scales @ true_w + 0.3 * rng_.standard_normal(n) + 2.0
        src = '内置真实统计量回退(等价)'
    # 标准化(随机特征对尺度敏感)
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)
    y = (y - y.mean()) / (y.std() + 1e-8)
    return X, y, src

Xc, yc, src = load_california_subset(n=40, d=6, seed=0)
print(f'数据来源: {src}; X{Xc.shape}, y{yc.shape}')
n_c = Xc.shape[0]

def rf_test_err_real(X, y, p, n_train, seed=0):
    rng_ = np.random.default_rng(seed)
    perm = rng_.permutation(X.shape[0])
    tr, te = perm[:n_train], perm[n_train:]
    W = rng_.standard_normal((X.shape[1], p)) * 0.8
    bb = rng_.uniform(0, 2*np.pi, size=p)
    Ptr = np.cos(X[tr] @ W + bb); Pte = np.cos(X[te] @ W + bb)
    w, *_ = np.linalg.lstsq(Ptr, y[tr], rcond=None)
    return np.mean((Pte @ w - y[te]) ** 2)

n_train = 30
ps = [3, 8, 15, 25, 30, 35, 45, 70, 120]
curve = [np.median([rf_test_err_real(Xc, yc, p, n_train, s) for s in range(25)]) for p in ps]
print(f"{'p':>6}{'测试MSE(中位)':>14}")
for p, m in zip(ps, curve):
    print(f'{p:>6}{m:>14.3f}{"  <- p≈n_train" if abs(p-n_train)<=5 else ""}')
peak = int(np.argmax(curve))
assert abs(ps[peak] - n_train) <= 10, '真实数据上尖峰也在 p≈训练样本数附近'
assert curve[-1] < curve[peak], '过参数化区也二次下降'
print('✅ 真实数据上同样出现双下降: 尖峰在 p≈n_train, 之后二次下降')

**🧪 胶囊练习**：实现 `peak_location(ps, curve)`：返回测试误差曲线的**峰值容量**与**峰值/末端误差比**(衡量二次下降的幅度)。

In [ ]:
def peak_location(ps, curve):
    # TODO: 返回 (峰值处的 p, curve峰值/curve末端)
    raise NotImplementedError

In [ ]:
# 自测
pk_p, ratio = peak_location(ps, curve)
assert pk_p == ps[int(np.argmax(curve))]
assert ratio > 1.0, '峰值应高于末端(存在二次下降)'
print(f'峰值容量 p={pk_p}, 峰值/末端 = {ratio:.2f}x (二次下降幅度)')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def peak_location(ps, curve):
    curve = np.asarray(curve)
    i = int(np.argmax(curve))
    return ps[i], float(curve[i] / curve[-1])

### 小结
- **偏差-方差分解**对平方损失恒成立; 方差只能在多套训练集上重复拟合来测; 经典图景是 **U 型**(欠参数化区)。
- **插值阈值**(容量≈样本数)处训练误差首次=0、测试误差**尖峰**; 越过它**二次下降** = double descent。
- 第二段下降的机制: 过参数化有**无穷多插值解**, 优化器**隐式偏向最小范数解**(更平滑) -> 方差回落、泛化更好。
- 双下降沿**容量/样本数/训练步数**三轴都出现(Nakkiran), 可用 EMC 统一; 显式 ridge 能抹平尖峰。
- 实践: 看到「加大模型/加数据/多训练反而短暂变差」, 先怀疑是不是路过了插值阈值, 别误判为 bug。

下一站: **模块 02 · Grokking 与涌现** —— 把「训练时间维」的反常推向极致: 训练早过拟合、测试很久后突然跳变。